# ValidEval V7.2 — GSM8K S1 engineering smoke

Canonical pre-Kaggle S1 notebook. It delegates all inference, resume, validation, import, and
packaging to tested package code. Real outputs are `ENGINEERING_ONLY`; local fixture outputs are
always `NON_EVIDENCE_FIXTURE` and cannot advance S1 acceptance or scientific claims.

In [ ]:
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path(os.environ.get("VALIDEVAL_REPOSITORY_ROOT", ".")).resolve()
STAGE = 'gsm8k'
MODE = os.environ.get("VALIDEVAL_EXECUTION_MODE", "fixture").strip().lower()
OUTPUT_ROOT = Path(os.environ.get("VALIDEVAL_NOTEBOOK_OUTPUT_ROOT", "kaggle_icml2027_outputs")).resolve()
REQUIREMENTS = ROOT / "requirements-kaggle-t4x2-v7-2-1.lock"
EXPECTED_REQUIREMENTS_SHA256 = 'efe3bb0e05f9df4fb7b9fde8bbab059003a7d5d3d58ab0cadf29b06433bab565'
assert hashlib.sha256(REQUIREMENTS.read_bytes()).hexdigest() == EXPECTED_REQUIREMENTS_SHA256
if MODE != "fixture" and importlib.util.find_spec("valideval") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", str(ROOT)], check=True)

from valideval.execution.notebook_v7_2 import run_notebook_stage_v7_2  # noqa: E402, I001

print(json.dumps({"stage": STAGE, "mode": MODE, "output_root": str(OUTPUT_ROOT)}, indent=2))

In [ ]:
RESULT = run_notebook_stage_v7_2(
    STAGE,
    mode=MODE,
    output_root=OUTPUT_ROOT,
    repository_root=ROOT,
)
print(json.dumps(RESULT, indent=2, sort_keys=True))

In [ ]:
EXPECTED_ZIPS = [
    OUTPUT_ROOT / "packages" / "valideval_v7_2_s1_mmlu_s1-v7-2-mmlu.zip",
    OUTPUT_ROOT / "packages" / "valideval_v7_2_s1_gsm8k_s1-v7-2-gsm8k.zip",
    OUTPUT_ROOT / "packages" / "valideval_v7_2_s1_bbh_s1-v7-2-bbh.zip",
]
print("Expected ZIP paths:")
for path in EXPECTED_ZIPS:
    print(path)
print("Resume: set VALIDEVAL_EXECUTION_MODE=resume and rerun the benchmark notebook.")
print("Exact local import/acceptance command:")
print("python -m valideval accept-s1-v7-2-1 --input-dir kaggle_icml2027_outputs/packages --output-root imported/v7_2_1/s1")
print("Post-acceptance recalibration command:")
print("python -m valideval recalibrate-study-c-after-s1 --input-root imported/v7_2_1/s1 --output results/final_cpu_maxout/planning/study_c_recalibration_after_s1.json")